# **Activate GPU**

In [ ]:
import tensorflow as tf
print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))


TensorFlow version: 2.19.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


# **Add Important Libraries**

In [ ]:
import os
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import ConfusionMatrixDisplay

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.preprocessing import image
from tensorflow.keras.models import Sequential,Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense,Dropout,Rescaling,RandomFlip,RandomRotation,RandomZoom,BatchNormalization,GlobalAveragePooling2D,RandomContrast,RandomBrightness,Activation


from tensorflow.keras.initializers import HeNormal
from tensorflow.keras import layers
from tensorflow.data import AUTOTUNE
from tensorflow.keras.optimizers import Adam
from keras.callbacks import EarlyStopping

from tensorflow.keras.models import load_model

# **Add Google Drive**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


# **Add Kaggle API**

In [ ]:
from google.colab import files
files.upload()



# **Add Dataset From Kaggle**

In [ ]:
!mkdir -p ~/.kaggle


!mv kaggle.json ~/.kaggle/


!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d rizwan123456789/potato-disease-leaf-datasetpld

!unzip -q potato-disease-leaf-datasetpld.zip -d ./data

!rm potato-disease-leaf-datasetpld.zip

!ls ./data

Dataset URL: https://www.kaggle.com/datasets/rizwan123456789/potato-disease-leaf-datasetpld
License(s): DbCL-1.0
  0% 0.00/37.4M [00:00<?, ?B/s]
100% 37.4M/37.4M [00:00<00:00, 1.67GB/s]
PLD_3_Classes_256


# **Add pretrained model ResNet50**

In [ ]:
from tensorflow.keras.applications import ResNet50
base_model_resnet=ResNet50(include_top=False,input_shape=(224,224,3),weights='imagenet')

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


# **Set Data Directory**

In [ ]:
data_dir = "/content/data/PLD_3_Classes_256"

train_dir = os.path.join(data_dir, "Training")
val_dir   = os.path.join(data_dir, "Validation")
test_dir  = os.path.join(data_dir, "Testing")

# **Split The Data into Train and Test Set**

In [ ]:
train_ds=keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=(224,224),
    batch_size=32,
    shuffle=True,
    color_mode="rgb",
    label_mode="categorical",
    labels="inferred"
)
val_ds=keras.utils.image_dataset_from_directory(
    val_dir,
    image_size=(224,224),
    batch_size=32,
    shuffle=False,
    color_mode="rgb",
    label_mode="categorical",
    labels="inferred"
)
test_ds=keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=(224,224),
    batch_size=32,
    shuffle=False,
    color_mode="rgb",
    label_mode="categorical",
    labels="inferred"
)

Found 3251 files belonging to 3 classes.
Found 416 files belonging to 3 classes.
Found 405 files belonging to 3 classes.


## **Find The Class Name**

In [ ]:
class_names=train_ds.class_names
print(class_names)

['Early_Blight', 'Healthy', 'Late_Blight']


# **Perform Cache and Autotune for Fast Processing**

In [ ]:
train_ds=train_ds.cache().shuffle(2500).prefetch(buffer_size=AUTOTUNE)
test_ds=test_ds.cache().prefetch(buffer_size=AUTOTUNE)
val_ds=val_ds.cache().prefetch(buffer_size=AUTOTUNE)

# **Perform Data Augmentation to Reduce Overfitting**

In [ ]:
data_augmentation = keras.Sequential(
    [
        RandomFlip("horizontal"),
        RandomRotation(0.1),
        RandomZoom(0.2),
    ]
)

# **Freeze Upper LAyer and Add Custom Dense Layer**

In [ ]:
#Freeze all layers
for layer in base_model_resnet.layers:
  layer.trainable = False


#unfreeze last 100 layers
for layer in base_model_resnet.layers[-50:]:
  layer.trainable=True



inp=layers.Input(shape=(224,224,3))

x=data_augmentation(inp)
x = base_model_resnet(x, training=False)

#add own fully connected layers

x=GlobalAveragePooling2D()(x)
x=Dense(128,activation="relu",kernel_initializer='he_normal')(x)
x=Dropout(0.4)(x)
x=Dense(64,activation="relu",kernel_initializer='he_normal')(x)
x=Dropout(0.4)(x)
output = Dense(3, activation="softmax")(x)



model_resnet=Model(inp,output)

# **Model Summary**

In [ ]:
model_resnet.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential_1 (Sequential)       │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 128)            │       262,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,858,435 (91.01 MB)

 Trainable params: 17,221,251 (65.69 MB)

 Non-trainable params: 6,637,184 (25.32 MB)

# **Compile The Model Using Adam Optimizer**

In [ ]:
model_resnet.compile(optimizer=Adam(learning_rate=1e-6),loss='categorical_crossentropy',metrics=['accuracy'])


# **Perform Early Stopping to Reduce Overfitting**

In [ ]:
from keras.callbacks import EarlyStopping
early_stopping = EarlyStopping(monitor='val_loss', patience=6,verbose=1, restore_best_weights=True)

# **Train Resnet Model**

In [ ]:
history_resnet=model_resnet.fit(train_ds,epochs=80,validation_data=val_ds,verbose=1,callbacks=[early_stopping])

Epoch 1/80
102/102 ━━━━━━━━━━━━━━━━━━━━ 52s 272ms/step - accuracy: 0.3788 - loss: 1.4866 - val_accuracy: 0.4663 - val_loss: 1.0233
Epoch 2/80
102/102 ━━━━━━━━━━━━━━━━━━━━ 24s 236ms/step - accuracy: 0.3866 - loss: 1.4084 - val_accuracy: 0.5433 - val_loss: 0.9146
Epoch 3/80
102/102 ━━━━━━━━━━━━━━━━━━━━ 24s 236ms/step - accuracy: 0.4361 - loss: 1.2321 - val_accuracy: 0.6082 - val_loss: 0.8305
Epoch 4/80
102/102 ━━━━━━━━━━━━━━━━━━━━ 24s 232ms/step - accuracy: 0.4715 - loss: 1.1783 - val_accuracy: 0.6635 - val_loss: 0.7624
Epoch 5/80
102/102 ━━━━━━━━━━━━━━━━━━━━ 24s 233ms/step - accuracy: 0.5005 - loss: 1.0985 - val_accuracy: 0.7139 - val_loss: 0.7032
Epoch 6/80
102/102 ━━━━━━━━━━━━━━━━━━━━ 24s 235ms/step - accuracy: 0.5278 - loss: 1.0101 - val_accuracy: 0.7692 - val_loss: 0.6527
Epoch 7/80
102/102 ━━━━━━━━━━━━━━━━━━━━ 24s 235ms/step - accuracy: 0.5589 - loss: 0.9569 - val_accuracy: 0.7861 - val_loss: 0.6104
Epoch 8/80
102/102 ━━━━━━━━━━━━━━━━━━━━ 24s 233ms/step - accuracy: 0.5801 - loss: 0

# **Save Our Resnet Model in Local Environment**

In [ ]:
model_resnet.save("potato_resnet_model.h5")


converter = tf.lite.TFLiteConverter.from_keras_model(model_resnet)
tflite_model_resnet = converter.convert()

with open("potato_resnet_model.tflite", "wb") as f:
    f.write(tflite_model_resnet)


Saved artifact at '/tmp/tmp9jlx5uzt'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_tensor_189')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  133611953002576: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133611953003344: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133611953002768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133611953003920: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133611953001040: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133611953001616: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133611953005456: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133611953006800: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133611953008720: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133611953008912: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1336119530

# **Load The Resnet Model**

In [ ]:
model_resnet = load_model(
    "/content/drive/MyDrive/cnn_full_2model_resnet.h5"
)

# **Check The Accuracy And Loss**

In [ ]:
resnet_loss,resnet_acc=model_resnet.evaluate(test_ds)
print()
print()
print("Accuracy: ",resnet_acc)
print()
print("Loss: ",resnet_loss)

13/13 ━━━━━━━━━━━━━━━━━━━━ 3s 206ms/step - accuracy: 0.9752 - loss: 0.0832


Accuracy:  0.9827160239219666

Loss:  0.06373564898967743


# **Evaluate Matrics**

In [ ]:
y_true = np.concatenate([y for x, y in test_ds], axis=0)
y_true = np.argmax(y_true, axis=1)

y_pred = np.argmax(model_resnet.predict(test_ds), axis=1)


print(" Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\n Classification Report:")
print(classification_report(
    y_true, y_pred,
    target_names=['Early Blight', 'Healthy', 'Late Blight']
))


13/13 ━━━━━━━━━━━━━━━━━━━━ 6s 296ms/step
 Confusion Matrix:
[[157   4   1]
 [  0 102   0]
 [  0   2 139]]

 Classification Report:
              precision    recall  f1-score   support

Early Blight       1.00      0.97      0.98       162
     Healthy       0.94      1.00      0.97       102
 Late Blight       0.99      0.99      0.99       141

    accuracy                           0.98       405
   macro avg       0.98      0.98      0.98       405
weighted avg       0.98      0.98      0.98       405



# **View Some Predictions**

In [ ]:
img_path = "/content/data/PLD_3_Classes_256/Testing/Early_Blight/Early_Blight_10.jpg"
img = image.load_img(img_path, target_size=(150, 150))
img_array = image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)


prediction = model.predict(img_array)
pred_index = np.argmax(prediction, axis=1)[0]
pred_class = class_names[pred_index]
confidence = np.max(prediction) * 100

print(f" Predicted: {pred_class} (Confidence: {confidence:.2f}%)")


plt.imshow(img)
plt.title(f"Predicted: {pred_class} ({confidence:.2f}%)")
plt.axis("off")
plt.show()

In [ ]:
img_path = "/content/data/PLD_3_Classes_256/Testing/Early_Blight/Early_Blight_10.jpg"
img = image.load_img(img_path, target_size=(150, 150))
img_array = image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)


prediction = model.predict(img_array)
pred_index = np.argmax(prediction, axis=1)[0]
pred_class = class_names[pred_index]
confidence = np.max(prediction) * 100

print(f" Predicted: {pred_class} (Confidence: {confidence:.2f}%)")


plt.imshow(img)
plt.title(f"Predicted: {pred_class} ({confidence:.2f}%)")
plt.axis("off")
plt.show()

In [ ]:
img_path = "/content/data/PLD_3_Classes_256/Testing/Early_Blight/Early_Blight_10.jpg"
img = image.load_img(img_path, target_size=(150, 150))
img_array = image.img_to_array(img)
img_array = np.expand_dims(img_array, axis=0)


prediction = model.predict(img_array)
pred_index = np.argmax(prediction, axis=1)[0]
pred_class = class_names[pred_index]
confidence = np.max(prediction) * 100

print(f" Predicted: {pred_class} (Confidence: {confidence:.2f}%)")


plt.imshow(img)
plt.title(f"Predicted: {pred_class} ({confidence:.2f}%)")
plt.axis("off")
plt.show()